# Experiment 008 — KL-Selective JRR (Windows / VS Code)

Local RTX GPU entry point. Before opening this notebook, follow `LOCAL_WINDOWS_VSCODE.md` and select the **Python (steering-repair CUDA)** kernel in VS Code.


In [ ]:
import os, sys, pathlib, subprocess, platform
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo = next((p for p in candidates if (p / 'pyproject.toml').exists() and (p / 'configs').exists()), None)
assert repo is not None, f'Open this notebook from the steering-manifold-repair repository. cwd={cwd}'
os.chdir(repo)
print('repo:', repo)
print('python:', sys.executable)
print('version:', platform.python_version())
print('git:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


## 1. GPU / CUDA sanity check

Do not continue on CPU: the oracle decoding loop is intentionally expensive.


In [ ]:
import torch, transformer_lens, transformers, pandas as pd, numpy as np
print('torch:', torch.__version__)
print('torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA is not available. Check LOCAL_WINDOWS_VSCODE.md before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print('VRAM GiB:', round(props.total_memory / 1024**3, 2))
print('TransformerLens:', getattr(transformer_lens, '__version__', 'unknown'))
print('Transformers:', transformers.__version__)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
torch.cuda.empty_cache()


## 2. Restore the frozen steering direction if needed


In [ ]:
direction = Path('results/sentiment_direction.pt')
if not direction.exists():
    print('Frozen direction missing; rebuilding established baseline direction...')
    subprocess.run([sys.executable, 'scripts/validate_sentiment_baseline.py', '--config', 'configs/baseline_sentiment_gpt2.yaml'], check=True)
else:
    print('Using frozen direction:', direction)


## 3. Fast unit tests


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_selective_jrr.py', 'tests/test_jrr.py'], check=True)


## 4. Real-model numerical preflight

This checks the KL gradient against a finite difference and verifies that the selected correction is orthogonal to transported `Jv`.


In [ ]:
subprocess.run([sys.executable, 'scripts/preflight_selective_jrr.py', '--config', 'configs/selective_jrr_gpt2.yaml'], check=True)


## 5. Calibration

No beta/layer sweep. The gate was frozen before the new hold-out is opened.


In [ ]:
subprocess.run([sys.executable, 'scripts/run_selective_jrr.py', '--config', 'configs/selective_jrr_gpt2.yaml', '--phase', 'calibration'], check=True)


In [ ]:
import json
from IPython.display import display, Image
cal = json.loads(Path('results/selective_jrr/calibration_summary.json').read_text())
print(json.dumps(cal, indent=2))
display(pd.read_csv('results/selective_jrr/calibration_same_alpha.csv'))
display(pd.read_csv('results/selective_jrr/calibration_aggregate.csv'))
display(Image(filename='results/selective_jrr/calibration_pareto.png'))


## 6. Fresh held-out evaluation

Runs only if the calibration gate passes. The evaluation uses the new prompt split and seeds `101/211`.


In [ ]:
if cal.get('go_to_new_heldout', False):
    print('Calibration gate PASSED. Running NEW held-out...')
    subprocess.run([sys.executable, 'scripts/run_selective_jrr.py', '--config', 'configs/selective_jrr_gpt2.yaml', '--phase', 'evaluation'], check=True)
else:
    print('Calibration gate FAILED. Do not use --force; stop and archive calibration as a negative result.')


In [ ]:
if Path('results/selective_jrr/evaluation_summary.json').exists():
    ev = json.loads(Path('results/selective_jrr/evaluation_summary.json').read_text())
    print(json.dumps(ev, indent=2))
    display(pd.read_csv('results/selective_jrr/evaluation_same_alpha.csv'))
    display(pd.read_csv('results/selective_jrr/evaluation_aggregate.csv'))
    display(Image(filename='results/selective_jrr/evaluation_pareto.png'))


## 7. Pack results


In [ ]:
import shutil
archive = shutil.make_archive(str(repo / 'selective_jrr_results_local'), 'zip', repo / 'results' / 'selective_jrr')
print('Created:', archive)
